# V8b — Within-Patient PDC Re-evaluated with Alarm Aggregation

**Purpose.** The original V8 (`V8_Within_Patient.ipynb`) implemented a
within-patient chronological 80/20 split on the V6 PDC features and reported
window-level FPR/h. This notebook re-runs the same split and the same LR/SVM
classifiers but evaluates with the literature-standard alarm rule (K=5 of
M=12 vote, 30-min refractory). Because evaluation is chronological within a
single patient, alarm aggregation respects the causal arrow of time exactly
as a deployed system would.

AUC and AUC-PR are threshold-free and identical to the original V8.


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

# Cell 0 — Imports
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# [path set by bootstrap] CODE_DIR = r"<repo>/Code"
sys.path.insert(0, CODE_DIR)

from config import (
    DATA_ROOT, FS, STEP_SEC, EXCLUDED_PATIENTS, RESULTS_DIR, RANDOM_SEED,
    INTERICTAL_MULTIPLIER, MAX_INTERICTAL_ABS,
    ALARM_K, ALARM_M, ALARM_REFRACTORY,
)
from metrics import evaluate_with_alarms

from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import roc_curve, roc_auc_score, average_precision_score

np.random.seed(RANDOM_SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'V8b — alarm K={ALARM_K}/M={ALARM_M}, refractory={ALARM_REFRACTORY*STEP_SEC//60}min')


V8b — alarm K=5/M=12, refractory=30min


In [2]:
# Cell 1 — Load PDC cache (mirrors V8 loader; chronological order preserved)
PDC_CACHE = Path(CODE_DIR) / 'cache_pdc_var20'
assert PDC_CACHE.exists()

patients_all = sorted([
    p for p in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, p))
    and p.startswith('chb') and p not in EXCLUDED_PATIENTS
])

pdc_data = {}
for pid in patients_all:
    pdir = PDC_CACHE / pid
    if not pdir.exists(): continue
    feat_p = pdir / 'features.npy'
    lab_p  = pdir / 'labels.npy'
    if not (feat_p.exists() and lab_p.exists()): continue
    X = np.load(feat_p).astype(np.float32)
    y = np.load(lab_p).astype(np.int8)
    n_pre = int((y==1).sum())
    if n_pre < 3:        # need ≥3 preictal for split
        continue
    pdc_data[pid] = (X, y)

patient_ids = sorted(pdc_data.keys())
print(f'Loaded {len(patient_ids)} patients (chronological order preserved)')


Loaded 21 patients (chronological order preserved)


In [3]:
# Cell 2 — Within-patient temporal split (matches V8 logic)
def within_patient_split(X, y, test_preictal_frac=0.30, min_test_pre=2):
    pre_idx = np.where(y == 1)[0]
    if len(pre_idx) < min_test_pre + 1:
        return None, None, None, None
    n_test_pre = max(min_test_pre, int(len(pre_idx) * test_preictal_frac))
    n_test_pre = min(n_test_pre, len(pre_idx) - 1)
    split_at = int(pre_idx[-n_test_pre])
    Xtr, ytr = X[:split_at], y[:split_at]
    Xte, yte = X[split_at:], y[split_at:]
    if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
        return None, None, None, None
    return Xtr, ytr, Xte, yte
print('Split function ready.')


Split function ready.


In [4]:
# Cell 3 — Per-patient train + evaluate (smoothing + alarm-level operational threshold)
# Post-processing follows Truong 2018 / Bandarabadi 2015 / Mormann 2007:
#   raw probs → moving-average smoothing
#             → operational threshold (Sens maximised s.t. ALARM-level FPR/h ≤ 1.0)
#             → K-of-M sliding-window vote → 30-min refractory suppression.
from alarm_postproc import smooth_probs, operational_threshold

SMOOTH_WINDOW = 10           # 10 windows ≈ 100 s (matches M=12 voting window)
TARGET_FPR_H  = 1.0          # Mormann 2007 clinical threshold (alarm-level)

def make_lr():
    return ('LR', Pipeline([('scl', StandardScaler()),
        ('clf', LogisticRegression(max_iter=400, solver='lbfgs',
            class_weight='balanced', C=1.0, random_state=RANDOM_SEED))]))

def make_svm():
    return ('SVM', Pipeline([('scl', StandardScaler()),
        ('clf', SVC(kernel='rbf', class_weight='balanced',
            C=1.0, probability=True, random_state=RANDOM_SEED))]))

MODEL_FACTORIES = [make_lr, make_svm]

def run_within_patient_alarm(factory, pdc_data, pids):
    name, _ = factory()
    print(f'\n== {name} within-patient alarm ({len(pids)} patients, smooth={SMOOTH_WINDOW}w, alarm-level FPR/h≤{TARGET_FPR_H}) ==')
    rows_alarm, rows_cmp = [], []
    t0 = time.time()
    for fi, pid in enumerate(pids, 1):
        X, y = pdc_data[pid]
        Xtr, ytr, Xte, yte = within_patient_split(X, y)
        if Xtr is None:
            continue
        _, pipe = factory()
        pipe.fit(Xtr, ytr)
        probs_raw = pipe.predict_proba(Xte)[:, 1]
        if len(np.unique(yte)) < 2: continue

        # --- post-processing pipeline ---
        probs    = smooth_probs(probs_raw, window=SMOOTH_WINDOW)
        threshold = operational_threshold(
            probs, yte, STEP_SEC, target_fpr_h=TARGET_FPR_H,
            K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
        )

        a = evaluate_with_alarms(
            y_true=yte, probs=probs, threshold=threshold,
            K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
            patient_id=pid,
        )
        a['model'] = name; a['threshold'] = threshold
        rows_alarm.append(a)

        # window-level reference (raw probs, raw Youden-J)
        fpr_a, tpr_a, thr = roc_curve(yte, probs_raw)
        thr_raw = float(np.clip(thr[np.argmax(tpr_a - fpr_a)], 0.05, 0.95))
        pred_w = (probs_raw >= thr_raw).astype(int)
        tp=int(((pred_w==1)&(yte==1)).sum()); fp=int(((pred_w==1)&(yte==0)).sum())
        tn=int(((pred_w==0)&(yte==0)).sum()); fn=int(((pred_w==0)&(yte==1)).sum())
        sens_w = tp/max(tp+fn,1)
        hours_int = (yte==0).sum() * STEP_SEC / 3600.0
        fpr_w = fp/max(hours_int, 1e-9)
        rows_cmp.append(dict(patient=pid, model=name,
            auc=a['auc'], auc_pr=a['auc_pr'], threshold=threshold,
            sens_window=sens_w, fpr_h_window=fpr_w,
            sens_alarm=a['sensitivity'], fpr_h_alarm=a['fpr_per_hour']))
        print(f'  [{fi:2d}/{len(pids)}] {pid}  AUC={a["auc"]:.3f}  '
              f'thr={threshold:.2f}  Sens(a)={a["sensitivity"]:.3f}  '
              f'FPR/h(w)={fpr_w:.1f} → FPR/h(a)={a["fpr_per_hour"]:.2f}')
    print(f'  {name} done in {(time.time()-t0)/60:.1f}min')
    return rows_alarm, rows_cmp

print('Pipeline ready (alarm-level operational threshold).')


Pipeline ready (alarm-level operational threshold).


In [5]:
# Cell 4 — Run
results_alarm, results_cmp = {}, {}
for fac in MODEL_FACTORIES:
    a, c = run_within_patient_alarm(fac, pdc_data, patient_ids)
    name = fac()[0]
    results_alarm[name] = pd.DataFrame(a)
    results_cmp[name]   = pd.DataFrame(c)



== LR within-patient alarm (21 patients, smooth=10w, alarm-level FPR/h≤1.0) ==
  [ 1/21] chb01  AUC=0.858  thr=0.05  Sens(a)=0.011  FPR/h(w)=38.6 → FPR/h(a)=0.00
  [ 2/21] chb02  AUC=0.817  thr=0.05  Sens(a)=0.011  FPR/h(w)=38.6 → FPR/h(a)=0.00
  [ 3/21] chb03  AUC=0.487  thr=0.20  Sens(a)=0.007  FPR/h(w)=32.4 → FPR/h(a)=0.00
  [ 4/21] chb04  AUC=0.351  thr=0.81  Sens(a)=0.000  FPR/h(w)=67.3 → FPR/h(a)=0.85
  [ 5/21] chb05  AUC=0.560  thr=0.05  Sens(a)=0.007  FPR/h(w)=64.3 → FPR/h(a)=0.00
  [ 6/21] chb06  AUC=0.358  thr=0.83  Sens(a)=0.000  FPR/h(w)=23.7 → FPR/h(a)=0.90
  [ 7/21] chb07  AUC=0.721  thr=0.05  Sens(a)=0.007  FPR/h(w)=12.9 → FPR/h(a)=0.00
  [ 8/21] chb08  AUC=0.630  thr=0.05  Sens(a)=0.009  FPR/h(w)=186.5 → FPR/h(a)=0.00
  [ 9/21] chb09  AUC=0.387  thr=0.43  Sens(a)=0.011  FPR/h(w)=32.8 → FPR/h(a)=0.98
  [10/21] chb10  AUC=0.817  thr=0.29  Sens(a)=0.007  FPR/h(w)=130.9 → FPR/h(a)=0.99
  [11/21] chb13  AUC=0.336  thr=0.85  Sens(a)=0.000  FPR/h(w)=36.4 → FPR/h(a)=0.00
  [12

In [6]:
# Cell 5 — Save and summarise
for name, df in results_alarm.items():
    df.to_csv(Path(RESULTS_DIR) / f'lopo_v8b_alarm_{name}.csv', index=False)
for name, df in results_cmp.items():
    df.to_csv(Path(RESULTS_DIR) / f'lopo_v8b_compare_{name}.csv', index=False)

print(f'{"Model":<6} {"AUC":>6} {"AUC-PR":>7} '
      f'{"Sens(w)":>8} {"FPR/h(w)":>10} {"Sens(a)":>8} {"FPR/h(a)":>10}')
print('-' * 65)
for name, df in results_cmp.items():
    print(f'{name:<6} {df.auc.mean():>6.3f} {df.auc_pr.mean():>7.3f} '
          f'{df.sens_window.mean():>8.3f} {df.fpr_h_window.mean():>10.1f} '
          f'{df.sens_alarm.mean():>8.3f} {df.fpr_h_alarm.mean():>10.2f}')

print('\nWithin-patient alarm aggregation respects chronological order.')
print('Outputs: lopo_v8b_alarm_{LR,SVM}.csv, lopo_v8b_compare_{LR,SVM}.csv')


Model     AUC  AUC-PR  Sens(w)   FPR/h(w)  Sens(a)   FPR/h(a)
-----------------------------------------------------------------
LR      0.565   0.540    0.424      102.2    0.007       0.29
SVM     0.557   0.523    0.596      146.3    0.007       0.24

Within-patient alarm aggregation respects chronological order.
Outputs: lopo_v8b_alarm_{LR,SVM}.csv, lopo_v8b_compare_{LR,SVM}.csv
